In [1]:
import sys
from pathlib import Path

# if notebook is in PRIN/notebooks, parent() is PRIN
project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root))
print("Added to sys.path:", project_root)

import os
from openai import OpenAI
from dotenv import load_dotenv  
from pprint import pprint

import json

from pydantic import BaseModel
from textwrap import dedent
from IPython.display import Math

from pathlib import Path

from utils.schema_json import ReportData

from pydantic import BaseModel
from time import perf_counter

Added to sys.path: C:\Users\lucat\PythonRepositories\PRIN


In [2]:
load_dotenv()  # Load environment variables from .env file

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

# Parametri

In [ ]:
SYSTEM_PROMPT_FILE_NAME = "system_prompt_final.txt"

TEST_DATA_FILE_NAME = "data_finetune_guido_openai_test.jsonl"

#MODEL = 'ft:gpt-4.1-nano-2025-04-14:luca-tramonti::CchIIIJt'
#MODEL = 'ft:gpt-4.1-nano-2025-04-14:luca-tramonti::Cee3mzk1:ckpt-step-233'  # Oversampling 3 chunks
MODEL = 'ft:gpt-4.1-nano-2025-04-14:luca-tramonti::CfCqIAIw:ckpt-step-174'  # Oversampling 2 chunks

#MODEL_NAME_SAVE = 'FT-gpt-4.1-nano-2025-04-14_TEMP_0.0'
MODEL_NAME_SAVE = 'FT-gpt-4.1-nano-2025-04-14_oversampling'
TEMPERATURE = 0.0
PREDICTIONS_FILE_NAME = "predictions"

In [4]:
#Carica system prompt da file
system_prompt_path = Path('../models/prompts/' + SYSTEM_PROMPT_FILE_NAME)

with open(system_prompt_path, "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

print(SYSTEM_PROMPT)

Sei un assistente medico radiologico esperto nella stadiazione del carcinoma rettale tramite Risonanza Magnetica (RM).

Il tuo compito è estrarre le caratteristiche strutturate dal referto RM fornito e mappare i dati nello schema JSON sottostante.

Regole di output rigorose:

1. Restituisci esclusivamente un oggetto JSON valido. Nessun testo introduttivo, spiegazione, codice o commento fuori dal JSON.
2. Rispetta esattamente i tipi e i valori consentiti per ciascun campo (vedi elenco sotto).
  - Se un campo è numerico ma nel testo non compare un valore, scrivi null.
  - Se un campo ha opzioni predefinite ma non è chiaramente determinabile, scrivi null.
3. I campi booleani all’interno di posizione, infiltrazione_organi_dettagli e sedi_linfonodi  devono contenere solo true o false.
4. Non aggiungere o inventare informazioni non presenti nel referto. Se un dato manca o è ambiguo, scrivi null.


Criteri di interpretazione ed estrazione:

Morfologia:
  - Se il referto menziona componenti ag

# Load Test Data

In [5]:
# Load training data from file
test_data_path = Path('../data/ft-dataset/' + TEST_DATA_FILE_NAME)

with open(test_data_path, "r", encoding="utf-8") as f:
    test_data = [json.loads(line) for line in f]

# Generazione

## Preliminari generazione

In [6]:
class ReportText(BaseModel):
    report_text: str

In [7]:
if True:
    display(ReportText.model_json_schema())

    test_report = ReportText(report_text=test_data[0]['messages'][1]['content'])
    print(test_report.report_text)

{'properties': {'report_text': {'title': 'Report Text', 'type': 'string'}},
 'required': ['report_text'],
 'title': 'ReportText',
 'type': 'object'}

IN CORRISPONDENZA DELLA PARETE LATERALE DESTRA DEL RETTO BASSO, DA SUBITO AL DI SOPRA DELLO SFINTERE ANALE INTERNO CON ESTENSIONE CRANIALE PER CIRCA 3,5 CM , È PRESENTE FORMAZIONE AGGETTANTE CHE OCCUPA 2/4 DELLA CIRCONFERENZA DEL LUME , SI PRESENTA MORFOLOGICAMENTE A ""SCODELLA"" ED INVIA DIGITAZIONI POLIPOIDI NEL MESORETTO DI DESTRA .
NEL MESORETTO DI DESTRA SONO INOLTRE PRESENTI DUE LINFONODI DEL DIAMETRO RISPETTIVAMENTE DI CIRCA 7 ED 8 MM ED UNO DEL DIAMTERO DI CIRCA 3 MM A CONTORNI SFUMATI SOSPETTI PER LOCALIZZAZIONE DI MALATTIA.
CONCLUSIONI : STADIAZIONE RM T3N1M0


In [8]:
# Funzione generatrice
def extract_data_from_report(report_text: str, system_prompt: str = SYSTEM_PROMPT, temperature: float = TEMPERATURE):
    response = client.responses.parse(
        model=MODEL,
        temperature=temperature,
        input=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': report_text},
            
        ],
        text_format=ReportData
    )
    return response

In [9]:
# Esempio
if False:
    result = extract_data_from_report(test_report.report_text)
    if False:
        print(result.output_text)
        print(result.output_parsed.model_dump(mode='json'))

## Generazione Risultati

In [10]:
testi_input = []
for r in test_data:
    for m in r['messages']:
        if m['role'] == 'user':
            testi_input.append(m['content'])

In [11]:
risultati = []
t1 = perf_counter()
for i, testo in enumerate(testi_input):
    print(f'...processando output {i}')
    output = extract_data_from_report(testo)
    if output is None:
        risultati.append('no output')
    else:
        risultati.append(output)
t2 = perf_counter()
print(f'Tempo totale: {int((t2-t1)//60)} minuti {int((t2-t1))%60} secondi')

...processando output 0
...processando output 1
...processando output 2
...processando output 3
...processando output 4
...processando output 5
...processando output 6
...processando output 7
...processando output 8
...processando output 9
...processando output 10
...processando output 11
...processando output 12
...processando output 13
...processando output 14
...processando output 15
...processando output 16
...processando output 17
...processando output 18
...processando output 19
...processando output 20
...processando output 21
...processando output 22
...processando output 23
...processando output 24
...processando output 25
...processando output 26
...processando output 27
Tempo totale: 1 minuti 41 secondi


In [12]:
if True:
    print(risultati[0].output_text)
    display(risultati[0].output_parsed.model_dump(mode='json'))

{"morfologia":"solido_polipoide","posizione":{"basso":true,"medio":false,"alto":false,"giunzione":false},"ore_inizio":null,"ore_fine":null,"spessore_parietale":null,"estensione_cranio_caudale":35,"distanza_oai":0.0,"riflessione_peritoneale_anteriore":null,"infiltrazione_tessuto_adiposo":"si_5mm_plus","infiltrazione_sfinteri":null,"infiltrazione_organi_extra":"no","infiltrazione_organi_dettagli":{"pavimento_pelvico":false,"altro":false},"coinvolgimento_riflessione_peritoneale":null,"coinvolgimento_fascia_mesorettale":null,"numero_linfonodi_non_conosciuto":false,"linfonodi_sospetti":0,"sedi_linfonodi":{"mesorettali":false,"rettali_superiori":false,"otturatori":false,"iliaci":false,"altro":false},"depositi_tumorali":"no","numero_depositi":0,"emvi_esteso":"no","stadio_T":"T3ab","stadio_N":"N1a","stadio_N1c":false,"mrf":"-","emvi":null,"metastasi":null}


{'morfologia': 'solido_polipoide',
 'posizione': {'basso': True,
  'medio': False,
  'alto': False,
  'giunzione': False},
 'ore_inizio': None,
 'ore_fine': None,
 'spessore_parietale': None,
 'estensione_cranio_caudale': 35,
 'distanza_oai': 0.0,
 'riflessione_peritoneale_anteriore': None,
 'infiltrazione_tessuto_adiposo': 'si_5mm_plus',
 'infiltrazione_sfinteri': None,
 'infiltrazione_organi_extra': 'no',
 'infiltrazione_organi_dettagli': {'pavimento_pelvico': False, 'altro': False},
 'coinvolgimento_riflessione_peritoneale': None,
 'coinvolgimento_fascia_mesorettale': None,
 'numero_linfonodi_non_conosciuto': False,
 'linfonodi_sospetti': 0,
 'sedi_linfonodi': {'mesorettali': False,
  'rettali_superiori': False,
  'otturatori': False,
  'iliaci': False,
  'altro': False},
 'depositi_tumorali': 'no',
 'numero_depositi': 0,
 'emvi_esteso': 'no',
 'stadio_T': 'T3ab',
 'stadio_N': 'N1a',
 'stadio_N1c': False,
 'mrf': '-',
 'emvi': None,
 'metastasi': None}

# Salva risultati

In [13]:
results_dicts = []
for r in risultati:
    results_dicts.append(
        {
            'model': MODEL,
            'temperature': TEMPERATURE,
            'min_p': 0,
            'prediction': r.output_text
        }
    )

In [14]:
with open(PREDICTIONS_FILE_NAME + '-' + MODEL_NAME_SAVE + '.jsonl', 'w', encoding='utf-8') as f:
    for r in results_dicts:
        f.write(json.dumps(r) + '\n')